In [1]:
def predict_dropout1():
  ''' This function should return a list with 'Y' or 'N'
    based on predicting if the samples in entry_dropout1.csv
    dropped out or not'''

In [2]:
def predict_dropout2():
  ''' This function should return a list with 'Y' or 'N'
    based on predicting if the samples in entry_dropout2.csv
    dropped out or not'''

In [3]:
def predict_dropout3():
  ''' This function should return a list with 'Y' or 'N'
    based on predicting if the samples in entry_dropout3.csv
    dropped out or not'''

In [4]:
def predict_dropout4():
  ''' This function should return a list with 'Y' or 'N'
    based on predicting if the samples in entry_dropout4.csv
    dropped out or not'''

In [5]:
# --- Imports and Preprocessing Utilities (from dropout_analysis_final.ipynb) ---
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Encoding maps
GRADE_MAP = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
YEAR_MAP = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}

# --- Helper: Preprocess input for each stage ---
def preprocess_input(df, stage):
    df = df.copy()
    # Encode categorical columns
    df['external_flag'] = df['External'].map({'Y': 1, 'N': 0})
    df['year_num'] = df['Year'].str.lower().map(YEAR_MAP)
    # Grade columns present in each stage
    for col in ['test 1', 'test 2', 'test 3', 'ind cw', 'group cw', 'final grade']:
        if col in df.columns:
            df[f'{col}_num'] = df[col].str.upper().map(GRADE_MAP)
    # Forum/office hour amortisation by stage
    if stage == 1:
        for c in ['forum Q', 'forum A', 'office hour visits']:
            if c in df.columns:
                df[f'{c}_s1'] = df[c] / 4
        features = ['external_flag', 'year_num', 'session 1', 'session 2', 'test 1_num']
        features += [f'{c}_s1' for c in ['forum Q', 'forum A', 'office hour visits'] if f'{c}_s1' in df.columns]
    elif stage == 2:
        for c in ['forum Q', 'forum A', 'office hour visits']:
            if c in df.columns:
                df[f'{c}_s2'] = df[c] / 2
        features = ['external_flag', 'year_num', 'session 1', 'session 2', 'session 3', 'session 4', 'test 1_num', 'test 2_num']
        features += [f'{c}_s2' for c in ['forum Q', 'forum A', 'office hour visits'] if f'{c}_s2' in df.columns]
    elif stage == 3:
        for c in ['forum Q', 'forum A', 'office hour visits']:
            if c in df.columns:
                df[f'{c}_s3'] = df[c] * 0.75
        features = ['external_flag', 'year_num', 'session 1', 'session 2', 'session 3', 'session 4', 'session 5', 'test 1_num', 'test 2_num', 'test 3_num']
        features += [f'{c}_s3' for c in ['forum Q', 'forum A', 'office hour visits'] if f'{c}_s3' in df.columns]
    elif stage == 4:
        for c in ['forum Q', 'forum A', 'office hour visits']:
            if c in df.columns:
                df[f'{c}_s4'] = df[c]
        features = ['external_flag', 'year_num', 'session 1', 'session 2', 'session 3', 'session 4', 'session 5', 'session 6', 'test 1_num', 'test 2_num', 'test 3_num', 'ind cw_num', 'group cw_num', 'final grade_num']
        features += [f'{c}_s4' for c in ['forum Q', 'forum A', 'office hour visits'] if f'{c}_s4' in df.columns]
    else:
        raise ValueError('Invalid stage')
    return df, features

# --- Dummy training data for demonstration (replace with real training in practice) ---
# In a real scenario, you would load the full training data and fit the models as in dropout_analysis_final.ipynb
# Here, we fit on the test CSV itself for demonstration only

def fit_stage_model(X, y, stage):
    if stage == 1:
        model = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.5)
    elif stage == 2:
        model = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced_subsample', random_state=42)
    elif stage == 3:
        model = HistGradientBoostingClassifier(max_iter=200, max_depth=5, random_state=42)
    elif stage == 4:
        model = StackingClassifier(
            estimators=[
                ('lr', LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced')),
                ('rf', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced_subsample')),
                ('hgb', HistGradientBoostingClassifier(max_iter=150, random_state=42)),
            ],
            final_estimator=LogisticRegression(C=1.0), passthrough=False, cv=3
        )
    else:
        raise ValueError('Invalid stage')
    model.fit(X, y)
    return model

# --- Prediction functions for each stage ---
def predict_dropout1():
    df = pd.read_csv('./data/entry_dropout1.csv')
    model, imp, scaler, features = stage_models[1]
    df_proc, features = preprocess_input(df, stage=1)
    X = imp.transform(df_proc[features])
    if scaler:
        X = scaler.transform(X)
    pred = model.predict(X)
    return ['Y' if p == 1 else 'N' for p in pred]

def predict_dropout2():
    df = pd.read_csv('./data/entry_dropout2.csv')
    model, imp, scaler, features = stage_models[2]
    df_proc, features = preprocess_input(df, stage=2)
    X = imp.transform(df_proc[features])
    pred = model.predict(X)
    return ['Y' if p == 1 else 'N' for p in pred]

def predict_dropout3():
    df = pd.read_csv('./data/entry_dropout3.csv')
    model, imp, scaler, features = stage_models[3]
    df_proc, features = preprocess_input(df, stage=3)
    X = imp.transform(df_proc[features])
    pred = model.predict(X)
    return ['Y' if p == 1 else 'N' for p in pred]

def predict_dropout4():
    df = pd.read_csv('./data/entry_dropout4.csv')
    model, imp, scaler, features = stage_models[4]
    df_proc, features = preprocess_input(df, stage=4)
    X = imp.transform(df_proc[features])
    if scaler:
        X = scaler.transform(X)
    pred = model.predict(X)
    return ['Y' if p == 1 else 'N' for p in pred]

# To use real models, load your training data, fit the models as in dropout_analysis_final.ipynb, and use model.predict(X) for each stage.

In [6]:
# Example usage:
if __name__ == "__main__":
    print("Predictions for Stage 1:", predict_dropout1())
    print("Predictions for Stage 2:", predict_dropout2())
    print("Predictions for Stage 3:", predict_dropout3())
    print("Predictions for Stage 4:", predict_dropout4())

NameError: name 'stage_models' is not defined

In [7]:
# --- Load training data and fit staged models ---

# Path to main training data (update if needed)
TRAIN_PATH = 'ND26_dropout.csv'
train_df = pd.read_csv(TRAIN_PATH)

# Helper to encode and engineer features for training

def prepare_training_data(stage):
    df = train_df.copy()
    df['external_flag'] = df['External'].map({'Y': 1, 'N': 0})
    df['year_num'] = df['Year'].str.lower().map(YEAR_MAP)
    for col in ['test 1', 'test 2', 'test 3', 'ind cw', 'group cw', 'final grade']:
        if col in df.columns:
            df[f'{col}_num'] = df[col].str.upper().map(GRADE_MAP)
    df['dropout_target'] = df['dropout'].map({'Y': 1, 'N': 0})
    # Forum/office hour amortisation by stage
    if stage == 1:
        for c in ['forum Q', 'forum A', 'office hour visits']:
            if c in df.columns:
                df[f'{c}_s1'] = df[c] / 4
        features = ['external_flag', 'year_num', 'session 1', 'session 2', 'test 1_num']
        features += [f'{c}_s1' for c in ['forum Q', 'forum A', 'office hour visits'] if f'{c}_s1' in df.columns]
    elif stage == 2:
        for c in ['forum Q', 'forum A', 'office hour visits']:
            if c in df.columns:
                df[f'{c}_s2'] = df[c] / 2
        mask = df['session 3'].notna() | df['test 2'].notna()
        df = df[mask]
        features = ['external_flag', 'year_num', 'session 1', 'session 2', 'session 3', 'session 4', 'test 1_num', 'test 2_num']
        features += [f'{c}_s2' for c in ['forum Q', 'forum A', 'office hour visits'] if f'{c}_s2' in df.columns]
    elif stage == 3:
        for c in ['forum Q', 'forum A', 'office hour visits']:
            if c in df.columns:
                df[f'{c}_s3'] = df[c] * 0.75
        mask = df['session 5'].notna() | df['test 3'].notna()
        df = df[mask]
        features = ['external_flag', 'year_num', 'session 1', 'session 2', 'session 3', 'session 4', 'session 5', 'test 1_num', 'test 2_num', 'test 3_num']
        features += [f'{c}_s3' for c in ['forum Q', 'forum A', 'office hour visits'] if f'{c}_s3' in df.columns]
    elif stage == 4:
        for c in ['forum Q', 'forum A', 'office hour visits']:
            if c in df.columns:
                df[f'{c}_s4'] = df[c]
        mask = df['session 6'].notna() | df['ind cw'].notna()
        df = df[mask]
        features = ['external_flag', 'year_num', 'session 1', 'session 2', 'session 3', 'session 4', 'session 5', 'session 6', 'test 1_num', 'test 2_num', 'test 3_num', 'ind cw_num', 'group cw_num', 'final grade_num']
        features += [f'{c}_s4' for c in ['forum Q', 'forum A', 'office hour visits'] if f'{c}_s4' in df.columns]
    else:
        raise ValueError('Invalid stage')
    X = df[features]
    y = df['dropout_target']
    return X, y

# Fit and store models for each stage
stage_models = {}
for stage in [1, 2, 3, 4]:
    X, y = prepare_training_data(stage)
    # Impute and scale as in prediction
    imp = SimpleImputer(strategy='median')
    X_imp = imp.fit_transform(X)
    scaler = StandardScaler() if stage in [1, 4] else None
    if scaler:
        X_imp = scaler.fit_transform(X_imp)
    model = fit_stage_model(X_imp, y, stage)
    stage_models[stage] = (model, imp, scaler, X.columns.tolist())

In [8]:
# Example usage:
if __name__ == "__main__":
    print("Predictions for Stage 1:", predict_dropout1())
    print("Predictions for Stage 2:", predict_dropout2())
    print("Predictions for Stage 3:", predict_dropout3())
    print("Predictions for Stage 4:", predict_dropout4())

Predictions for Stage 1: ['Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y']
Predictions for Stage 2: ['Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y']
Predictions for Stage 3: ['Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y', 'Y']
Predictions for Stage 4: ['N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N']
